# Uncertainty-Aware Traffic Routing Using Search-Based Algorithms

---

## Problem Statement

Traditional route-planning systems typically compute a single shortest path assuming fixed and deterministic travel times. However, real-world traffic conditions are inherently uncertain due to congestion and incidents. Ignoring this uncertainty can lead to routes that are optimal on average but unreliable in practice. The goal of this project is to design and implement an AI-based traffic routing system that incorporates search algorithms and uncertainty modeling to compute routes that are efficient and reliable.

> **Note:** AI assistance was used in generating code comments and documentation throughout this project.

## Phase 1: Dataset Construction

### Overview

The dataset was manually constructed to represent a real world road network across New York State. It is structured as a **graph**, where cities are nodes and roads connecting them are edges. The data is split into two CSV files.

---

### File 1: `ny_graph_nodes.csv` -> Cities (Nodes)

This file contains **27 cities** across New York State, each representing a node in the traffic graph.

| Column | Description |
|---|---|
| `city` | Name of the city |
| `population` | Population of the city |
| `region` | Geographic region (e.g., NYC Metro, Western NY) |
| `lat` | Latitude coordinate |
| `lng` | Longitude coordinate |

---

### File 2: `ny_graph_edges.csv` —> Roads (Edges)

This file contains **41 road connections** between cities, each representing an edge in the traffic graph.

| Column | Description |
|---|---|
| `from_city` | Origin city |
| `to_city` | Destination city |
| `highway` | Highway/road name (e.g., I-90, I-87) |
| `miles` | Distance in miles |
| `speed_mph` | Speed limit in mph |
| `road_type` | Type of road (`interstate`, `state route`, `parkway`) |
| `am_peak_mean` | Average congestion multiplier during AM peak hours |
| `am_peak_std` | Standard deviation of AM peak congestion |
| `off_peak_mean` | Average congestion multiplier during off-peak hours |
| `off_peak_std` | Standard deviation of off-peak congestion |
| `pm_peak_mean` | Average congestion multiplier during PM peak hours |
| `pm_peak_std` | Standard deviation of PM peak congestion |
| `free_flow_time_min` | Travel time in minutes under free-flow (no traffic) conditions |

---

### Uncertainty Modeling

Each edge includes **congestion multipliers** with a mean and standard deviation for three time periods:

- **AM Peak** (morning rush hour) —> highest uncertainty in urban areas
- **Off Peak** (midday / night) —> lowest congestion and uncertainty
- **PM Peak** (evening rush hour) —> highest congestion overall

The congestion multiplier is applied to `free_flow_time_min` to simulate realistic travel times.

---

In [16]:
# Phase 1 - Data Structures and Graph Construction

import math
import random
import heapq
import networkx as nx
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Dict, Optional

@dataclass
class TrafficDistribution:
    """
    Represents the uncertainty in travel time for a given time period. Uses a mean and standard deviation (Gaussian model) as the traffic multiplier.
    
    Atttributes:
        mean (float): The average traffic multiplier
        std (float): The standard deviation of the traffic multiplier, representing uncertainty.
    """
    __slots__ = ['mean', 'std']
    mean: float
    std: float

    def sample(self) -> float:
        """
        Sample a traffic multiplier using a Gaussian distribution.
        
        :return: A traffic multiplier sampled from the distribution, ensuring it's at least 1.0 since traffic cannot be faster than free-flow speed.
        """
        return max(1.0, random.gauss(self.mean, self.std))

    def __repr__(self):
        """
        String representation of the traffic distribution for debugging.
        
        :return: A string showing the mean and standard deviation of the traffic distribution.
        """
        return f"TrafficDistribution(mean={self.mean}, std={self.std})"

@dataclass
class Edge:
    """
    Represents a road (edge) connecting two cities in the traffic graph.
    
    Attributes:
        from_city (str): The starting city of the road.
        to_city (str): The destination city of the road.
        highway (str): The name or number of the highway.
        miles (float): The length of the road in miles.
        speed_mph (float): The free flow speed in miles per hour.
        road_type (str): The type of road (e.g., 'interstate', 'state route').
        am_peak (TrafficDistribution): Traffic profile for morning rush hour.
        off_peak (TrafficDistribution): Traffic profile for midday/night.
        pm_peak (TrafficDistribution): Traffic profile for evening rush hour.
        free_flow_time_min (float): The travel time in minutes under free flow conditions, calculated as (miles / speed_mph) * 60.
    """
    __slots__ = ['from_city', 'to_city', 'highway', 'miles', 'speed_mph', 'road_type', 'am_peak', 'off_peak', 'pm_peak', 'free_flow_time_min']
    from_city: str
    to_city: str
    highway: str
    miles: float
    speed_mph: float
    road_type: str
    am_peak: TrafficDistribution
    off_peak: TrafficDistribution
    pm_peak: TrafficDistribution
    free_flow_time_min: float

    def get_travel_time(self, time_period: str, sampled: bool = False) -> float:
        """
        Returns travel time in minutes for a given time period.

        :param time_period: One of 'am_peak', 'off_peak', 'pm_peak'
        :param sampled: If True, sample from the traffic distribution; otherwise, use the mean multiplier.
        :return: Travel time in minutes, rounded to 2 decimal places.
        """
        distributions = {
            'am_peak': self.am_peak,
            'off_peak': self.off_peak,
            'pm_peak': self.pm_peak
        }

        if time_period not in distributions:
            raise ValueError(f"Invalid time_period '{time_period}'. Choose from: {list(distributions.keys())}")

        profile = distributions[time_period]

        if sampled:
            multiplier = profile.sample()
        else:
            multiplier = profile.mean

        return round(self.free_flow_time_min * multiplier, 2)

    def get_expected_travel_time_with_probabilistic_traffic(self, time_period: str) -> float:
        """
        Returns the sampled (uncertain) travel time in minutes for the given time period, drawn from the traffic distribution using a Gaussian model.

        :return: The sampled travel time in minutes, calculated using a randomly drawn traffic multiplier.
        """
        return self.get_travel_time(time_period, sampled=True)

    def get_expected_travel_time_with_traffic(self, time_period: str) -> float:
        """
        Returns the deterministic (mean) travel time in minutes for the given time period, using the average traffic multiplier with no randomness.
        
        Returns:
            The expected travel time in minutes, calculated using the mean traffic multiplier.
        """
        return self.get_travel_time(time_period, sampled=False)

    def get_travel_time_without_traffic(self) -> float:
        """
        Returns the free flow travel time in minutes, assuming zero traffic.
        
        :return: The free flow travel time in minutes with no traffic applied.
        """
        return self.free_flow_time_min

    def __repr__(self):
        return f"Edge({self.from_city} → {self.to_city} via {self.highway}, {self.miles} mi)"

@dataclass
class Node:
    """
    Represents a city (node) in the traffic graph.

    Attributes:
        city (str): The name of the city.
        population (int): The population of the city.
        region (str): The region of New York the city is located in.
            The cities are grouped into 5 regions:
                - NYC Metro — New York City, Bronx, Yonkers, Long Island City, etc.
                - Western NY — Buffalo, Rochester, Niagara Falls
                - Central NY — Syracuse, Utica, Rome
                - Capital Region — Albany, Schenectady, Troy, Saratoga Springs
                - Southern Tier — Binghamton
        lat (float): The latitude of the city, used for distance calculations.
        lng (float): The longitude of the city, used for distance calculations. 
    """
    __slots__ = ['city', 'population', 'region', 'lat', 'lng']
    city: str
    population: int
    region: str
    lat: float
    lng: float

    def haversine_distance(self, other: 'Node') -> float:
        """
        Computes straight line distance in miles to another node accounting for the earth's curvature.
        
        Useful as a heuristic for A* search. Why?
            - The Haversine distance gives us a lower bound on the actual travel distance between two cities, since roads cannot be perfectly straight and may have to navigate around terrain, infrastructure, etc.
            - This makes it an admissible heuristic for A* search, ensuring that it never overestimates the true cost to reach the goal, which is crucial for guaranteeing optimality of the A* algorithm.

        :param other: Another Node to which the distance is calculated.
        :return: The straight line distance in miles, rounded to 2 decimal places.
        """
        R = 3958.8 # This is the radius of the Earth in miles
        lat1, lat2 = math.radians(self.lat), math.radians(other.lat)
        dlat = math.radians(other.lat - self.lat)
        dlng = math.radians(other.lng - self.lng)
        a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlng / 2)**2
        return round(2 * R * math.asin(math.sqrt(a)), 2)

    def __repr__(self):
        """
        String representation of the Node for debugging.
        
        :return: A string showing the city name, region, and population of the node.
        """
        return f"Node({self.city}, region={self.region}, pop={self.population:,})"

@dataclass
class TrafficGraph:
    """
    Represents the full NY road network as an adjacency list graph. Nodes are cities, edges are roads with uncertainty-aware travel times.

    Attributes:
        nodes: A dictionary mapping city names to Node objects.
        edges: A list of all Edge objects in the graph.
        adjacency: A dictionary mapping city names to a list of outgoing Edge objects for efficient neighbor lookups.
    """
    __slots__ = ['nodes', 'edges', 'adjacency']
    nodes: Dict[str, Node]
    edges: List[Edge]
    adjacency: Dict[str, List[Edge]]

    def __init__(self):
        self.nodes: Dict[str, Node] = {}
        self.edges: List[Edge] = []
        self.adjacency: Dict[str, List[Edge]] = {}

    def load_nodes(self, filepath: str):
        """
        Load nodes from ny_graph_nodes.csv

        :param filepath: The path to the CSV file containing node data.
        :return: None
        """
        with open(filepath, 'r') as f:
            lines = f.read().splitlines()

        for line in lines[1:]:
            city, population, region, lat, lng = line.split(',')
            node = Node(
                city=city,
                population=int(population),
                region=region,
                lat=float(lat),
                lng=float(lng)
            )
            
            self.nodes[node.city] = node
            self.adjacency[node.city] = []

        print(f"Loaded {len(self.nodes)} nodes.")

    def load_edges(self, filepath: str):
        """
        Load edges from ny_graph_edges.csv

        :param filepath: The path to the CSV file containing edge data.
        :return: None
        """
        with open(filepath, 'r') as f:
            lines = f.read().splitlines()

        for line in lines[1:]:
            from_city, to_city, highway, miles, speed_mph, road_type, am_peak_mean, am_peak_std, off_peak_mean, off_peak_std, pm_peak_mean, pm_peak_std, free_flow_time_min = line.split(',')
            
            forward_edge = Edge(
                from_city=from_city,
                to_city=to_city,
                highway=highway,
                miles=float(miles),
                speed_mph=float(speed_mph),
                road_type=road_type,
                am_peak=TrafficDistribution(float(am_peak_mean), float(am_peak_std)),
                off_peak=TrafficDistribution(float(off_peak_mean), float(off_peak_std)),
                pm_peak=TrafficDistribution(float(pm_peak_mean), float(pm_peak_std)),
                free_flow_time_min=float(free_flow_time_min)
            )

            reverse_edge = Edge(
                from_city=to_city,
                to_city=from_city,
                highway=highway,
                miles=float(miles),
                speed_mph=float(speed_mph),
                road_type=road_type,
                am_peak=TrafficDistribution(float(am_peak_mean), float(am_peak_std)),
                off_peak=TrafficDistribution(float(off_peak_mean), float(off_peak_std)),
                pm_peak=TrafficDistribution(float(pm_peak_mean), float(pm_peak_std)),
                free_flow_time_min=float(free_flow_time_min)
            )

            self.edges.append(forward_edge)
            self.edges.append(reverse_edge)

            if forward_edge.from_city in self.adjacency:
                self.adjacency[forward_edge.from_city].append(forward_edge)
            if reverse_edge.from_city in self.adjacency:
                self.adjacency[reverse_edge.from_city].append(reverse_edge)

        print(f"Loaded {len(self.edges)} edges.")

    def get_node(self, city: str) -> Optional[Node]:
        """
        Get a node by city name.

        :param city: The name of the city to retrieve.
        :return: The Node object corresponding to the city, or None if not found.
        """
        return self.nodes.get(city)

    def get_neighbors(self, city: str) -> List[Edge]:
        """
        Get all edges from a given city.

        :param city: The name of the city to retrieve neighbors for.
        :return: A list of Edge objects representing the neighbors.
        """
        return self.adjacency.get(city, [])

    def get_edges_between(self, city_a: str, city_b: str) -> List[Edge]:
        """
        Get all road options between two cities.

        :param city_a: The name of the first city.
        :param city_b: The name of the second city.
        :return: A list of Edge objects representing the roads between the cities.
        """
        edges_between = []

        for edge in self.adjacency.get(city_a, []):
            if edge.to_city == city_b or edge.from_city == city_b:
                edges_between.append(edge)

        return edges_between

    def __repr__(self):
        """
        String representation of the TrafficGraph for debugging.
        
        :return: A string showing the number of nodes and edges in the graph.
        """
        return f"TrafficGraph(nodes={len(self.nodes)}, edges={len(self.edges)})"

In [17]:
def uniform_cost_search(graph, start, goal):
    
    priority_queue = [(0, start)]
    # Dictionary to store the cost of the shortest path to each node
    visited = {start: (0, None)}
    
    while priority_queue:
        # Pop the node with the lowest cost from the priority queue
        current_cost, current_node = heapq.heappop(priority_queue)
        
        # If we reached the goal, return the total cost and the path
        if current_node == goal:
            return current_cost, reconstruct_path(visited, start, goal)
        
        # Explore the neighbors
        for neighbor, cost in graph[current_node]:
            total_cost = current_cost + cost
            # Check if this path to the neighbor is better than any previously found
            if neighbor not in visited or total_cost < visited[neighbor][0]:
                visited[neighbor] = (total_cost, current_node)
                heapq.heappush(priority_queue, (total_cost, neighbor))
    
    return None

In [18]:
def a_star_search(graph, start, goal):

    heuristic_function = start.haversine_distance(goal) #using the haversine_distance function as a heuristic
    
    priority_queue = [(0 + heuristic_function, start)]
    # Dictionary to store the cost of the shortest path to each node
    visited = {start: (0, None)}
    
    while priority_queue:
        # Pop the node with the lowest cost from the priority queue
        current_cost, current_node = heapq.heappop(priority_queue)

        # If we reached the goal, return the total cost and the path
        if current_node == goal:
            return current_cost, reconstruct_path(visited, start, goal)
        
        # Explore the neighbors
        for neighbor, cost in graph[current_node]:
            total_cost = (current_cost - current_node.haversine_distance(goal)) + cost
            total_cost += neighbor.haversine_distance(goal)
            # Check if this path to the neighbor is better than any previously found
            if neighbor not in visited or total_cost < visited[neighbor][0]:
                visited[neighbor] = (total_cost, current_node)
                heapq.heappush(priority_queue, (total_cost, neighbor))
    
    return None

In [19]:
def reconstruct_path(visited, start, goal):
    # Reconstruct the path from start to goal by following the visited nodes
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = visited[current][1]  # Get the parent node
    path.reverse()
    return path

In [20]:
def visualize_graph(graph, path=None):
    G = nx.DiGraph()

    # Adding nodes and edges to the graph
    for node, edges in graph.items():
        for neighbor, cost in edges:
            G.add_edge(node, neighbor, weight=cost)

    pos = nx.spring_layout(G)  # Positioning the nodes

    # Drawing the graph
    plt.figure(figsize=(8, 6))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000, font_size=15, font_weight='bold', edge_color='gray')
    labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=12)

    if path:
        # Highlight the path in red
        path_edges = list(zip(path, path[1:]))
        nx.draw_networkx_edges(G, pos, edgelist=path_edges, edge_color='red', width=2.5)

    plt.title("Uniform Cost Search Path Visualization")
    plt.show()

In [21]:
traffic_graph = TrafficGraph()
traffic_graph.load_nodes('data/CSCI 630 - Artificial Intelligence - Project Data - ny_graph_nodes.csv')
traffic_graph.load_edges('data/CSCI 630 - Artificial Intelligence - Project Data - ny_graph_edges.csv')

Loaded 28 nodes.
Loaded 84 edges.
